# OOHScout — F5 Adaptation: DEV_MODE Test BBox for Waco Urban Core

**Current scope:** F5 — define a small bounding box around downtown Waco, clip F1 boundary + F2 corridor to it, prove the pipeline runs meaningfully faster.

**Why:** so future features (F6 buffer, F7 candidates, F10 POIs, etc.) can flip a flag to iterate on ~50 km² instead of the full 2,747 km² county. Faster feedback loop while debugging.

**What this notebook does:**
1. Loads F1's McLennan boundary and F2's IH-35 corridor.
2. Defines a Waco urban bounding box.
3. Clips both layers to the bbox.
4. Times the full-county vs bbox pipelines.
5. Plots them side-by-side on the Esri basemap.

## 1. Imports + repo-root anchor

In [ ]:
import time
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box

from oohscout.track_a_spatial import (
    load_or_build_study_area,
    load_or_build_highway_centerline,
)

working_dir = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in [working_dir, *working_dir.parents] if (p / 'pyproject.toml').exists()),
    None,
)
assert REPO_ROOT is not None
DATA_DIR = REPO_ROOT / 'backend' / 'data' / 'processed'

PLACE = 'McLennan County, Texas'
CRS_METRIC = 32614
CRS_GEOGRAPHIC = 4326

## 2. Load F1 + F2 outputs (full county)

In [ ]:
t0 = time.perf_counter()
study = load_or_build_study_area(PLACE, CRS_METRIC, DATA_DIR, reference_total_area_km2=2746.0)
corridor = load_or_build_highway_centerline(study.admin_poly, DATA_DIR, crs_metric=CRS_METRIC)
full_load_secs = time.perf_counter() - t0

full_area_km2 = study.study_area.area / 1e6
full_length_km = corridor.total_length_m / 1000
full_segments = len(corridor.corridor_gdf_metric)

print(f'Full county   : area {full_area_km2:,.1f} km², IH-35 {full_length_km:,.1f} km, {full_segments} segments')
print(f'Load time     : {full_load_secs:.2f} s')

## 3. Define the Waco urban bbox

This bbox was chosen to cover downtown Waco + Baylor University + the IH-35 mainline through the city — roughly 15 km × 15 km. Small enough to iterate quickly, big enough to contain interesting spatial variation.

In [ ]:
# (minx, miny, maxx, maxy) in WGS 84 degrees
WACO_URBAN_BBOX = (-97.20, 31.48, -97.05, 31.62)

# Convert to a Shapely polygon in the metric CRS so we can intersect against F1/F2 output.
bbox_wgs84 = gpd.GeoSeries([box(*WACO_URBAN_BBOX)], crs=CRS_GEOGRAPHIC)
bbox_metric = bbox_wgs84.to_crs(epsg=CRS_METRIC).iloc[0]

bbox_area_km2 = bbox_metric.area / 1e6
print(f'Waco bbox     : {bbox_area_km2:.1f} km² ({bbox_area_km2/full_area_km2*100:.1f}% of full county)')

## 4. Clip F2 corridor to the bbox

Use `intersects` (not `within`) so highway segments crossing the bbox boundary are kept whole. Time the clip separately from the load.

In [ ]:
t0 = time.perf_counter()
corridor_clipped = corridor.corridor_gdf_metric[
    corridor.corridor_gdf_metric.intersects(bbox_metric)
].reset_index(drop=True)
clip_secs = time.perf_counter() - t0

clipped_length_km = corridor_clipped.geometry.length.sum() / 1000
print(f'Clipped IH-35 : {clipped_length_km:.1f} km ({len(corridor_clipped)} segments)')
print(f'Reduction     : {full_length_km/clipped_length_km:.1f}x shorter, {full_segments/max(len(corridor_clipped),1):.1f}x fewer segments')
print(f'Clip time     : {clip_secs*1000:.1f} ms')

## 5. Visual comparison — full county vs Waco bbox

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# LEFT — full county
study.admin_gdf_metric.plot(ax=ax1, facecolor='none', edgecolor='#0a84ff', linewidth=1.5)
corridor.corridor_gdf_metric.plot(ax=ax1, color='#ff2d55', linewidth=1)
cx.add_basemap(ax1, crs=corridor.corridor_gdf_metric.crs, source=cx.providers.Esri.WorldImagery)
ax1.set_title(f'Full McLennan County — {full_area_km2:,.0f} km²', fontsize=12)
ax1.set_aspect('equal')
ax1.axis('off')

# RIGHT — Waco bbox only
gpd.GeoSeries([bbox_metric], crs=f'EPSG:{CRS_METRIC}').plot(
    ax=ax2, facecolor='none', edgecolor='#0a84ff', linewidth=2
)
corridor_clipped.plot(ax=ax2, color='#ff2d55', linewidth=1.5)
cx.add_basemap(ax2, crs=corridor_clipped.crs, source=cx.providers.Esri.WorldImagery)
ax2.set_title(f'DEV_MODE Waco bbox — {bbox_area_km2:.0f} km²', fontsize=12)
ax2.set_aspect('equal')
ax2.axis('off')

plt.tight_layout()
check_png = DATA_DIR / 'waco_dev_bbox_f5_check.png'
fig.savefig(check_png, dpi=120)
print(f'Saved comparison plot to {check_png}')
plt.show()

## F5 completion gate

F5 ships when:

1. ✅ `WACO_URBAN_BBOX` constant defined with clear reasoning
2. ✅ Bbox correctly projected to metric CRS before intersection
3. ✅ Clipped IH-35 is a proper subset of full IH-35 (fewer segments, shorter length)
4. ✅ Visual side-by-side confirms the bbox covers downtown Waco + IH-35 mainline
5. ⏳ Production module `backend/src/oohscout/track_a_spatial/dev_mode.py` exports the constant + a `clip_to_bbox()` helper
6. ⏳ Pytest verifies clipped output is smaller than full and passes CRS invariants

Items 1-4 you just did. Items 5-6 are what I'll build after you say "done learning".